# 03 - Grade the forgeries (Kaggle)

Session 3 of the plan (P8-P10). Notebook 02 made ~100 forged pages; now we measure how good they are:
**3** did the text come out right, and was the style kept (OCR accuracy, NED, SSIM, PSNR) - **4** the 2x2 grid -
**5** a blind test where YOU guess real or fake - **6** tag the 8 worst failures - **7** can Week 1 tools (ELA, SRM)
see the edit - **8** headline numbers and a zip to download.

### Before you run anything (Kaggle Settings panel)
1. **Accelerator:** GPU T4 x2 (EasyOCR is much faster on a GPU; CPU also works, just slower). 2. **Internet:** On.
3. **Add Input:** your uploaded notebook-02 output (the `working` dataset) **and** `PATH_TO_DATA500_DATASET` (the blind test needs fresh,
untouched words from pages we did not forge).

### Run it INTERACTIVELY (not Save & Run All)
Steps 5 and 6 ask you to type answers into a box. Run the notebook cell by cell in the editor. At the end, download
the zip from the file browser (folder icon > `working`), the same way you did for notebook 02.
Every step saves its answers, so re-running never makes you redo the blind test.

## Step 0 - CONFIG
All settings for the evaluation live here: the seed, where the inputs are, how many items go into the blind test and
the failure gallery, and the ELA settings. They are saved as `eval_config.json` in the run folder (next to notebook 02's
`config.json`, which we do not overwrite), so the evaluation can be repeated exactly. **Run this cell first.**

In [ ]:
# import random so we can fix Python's own random numbers
import random
# import numpy so we can fix its random numbers too
import numpy as np

# the seed: the same seed gives the same "random" choices every run
SEED = 42

# ---------- where things are ----------
# the folder where Kaggle puts the datasets we attached (read-only)
INPUT_ROOT = "/kaggle/input"
# the folder where Kaggle mounts OUR datasets (it includes the Kaggle username)
DATASETS_ROOT = INPUT_ROOT + "/datasets/PATH_TO_KAGGLE_USERNAME"
# the PATH_TO_DATA500_DATASET dataset (needed for untouched words in the blind test)
DATA_ROOT = DATASETS_ROOT + "/PATH_TO_DATA500_DATASET"
# the folder we are allowed to write to
WORK_DIR = "/kaggle/working"

# ---------- word filter (the same as notebook 02), used for the real crops in the blind test ----------
# 3 to 8 letters or digits
WORD_REGEX = r"^[A-Za-z0-9]{3,8}$"
# EasyOCR must be at least 70% sure
MIN_CONFIDENCE = 0.7
# the box must be at least 12 pixels tall
MIN_BOX_HEIGHT = 12
# extra pixels around each crop
CROP_PADDING = 3

# ---------- P9 settings ----------
# how many fake crops and how many real crops go into the blind test (10 + 10 = 20)
N_BLIND_EACH = 10
# how many of the worst forgeries we tag by hand
N_WORST = 8
# the failure types to choose from, numbered 1 to 6 when you tag
FAILURE_TYPES = ["wrong text", "blurry", "colour/background mismatch", "squashed long word", "box misaligned", "other"]
# True = Steps 5 and 6 ask you to type answers; set False if you ever use Save & Run All (they will be skipped)
INTERACTIVE = True

# ---------- P10 settings (ELA and SRM) ----------
# how many pages to show in the ELA / SRM figure
N_ELA_EXAMPLES = 4
# the JPEG quality used to re-save pages for ELA
ELA_QUALITY = 90
# how much to brighten the ELA picture so our eyes can see it (display only)
ELA_AMPLIFY = 20
# an ELA ratio above this means the edit "lights up"
ELA_RATIO_THRESHOLD = 1.5
# extra pixels shown around the edited box when we zoom in
ZOOM_MARGIN = 120

# fix Python's random generator
random.seed(SEED)
# fix numpy's random generator
np.random.seed(SEED)
# confirm the settings loaded
print("CONFIG loaded. SEED =", SEED, "| INTERACTIVE =", INTERACTIVE)

## Step 1 - Find notebook 02's run folder and copy it somewhere we can write
Kaggle inputs are read-only, but P8 asks us to ADD columns to `results.csv` and save new figures into `figures/`. So we
find the run folder inside your uploaded dataset (we search for `results.csv`, because the exact folder path depends
on how Kaggle unpacked the upload), copy it to `/kaggle/working/runs/`, and work on that copy. If the upload only
holds the zip from Step 13, the cell unzips it first. We also find the `PATH_TO_DATA500_DATASET` pages for the blind test.

**Expected output:** `Run folder found: ...run_YYYYMMDD_HHMM`, a file count for each sub-folder (about 100 in
`tampered/`, `masks/box/`, ...), the `PATH_TO_DATA500_DATASET` pages folder, and whether a GPU is available.

In [ ]:
# import os so we can search folders
import os
# import json so we can save the evaluation settings
import json
# import shutil so we can copy the run folder
import shutil
# import zipfile so we can unpack a zipped run if needed
import zipfile
# import torch so we can check for a GPU
import torch


# define a helper that lists every folder (up to max_depth levels down) that contains a given file name
def folders_containing(root, file_name, max_depth):
    # start an empty list
    found = []
    # walk through every folder under root
    for folder, subfolders, files in os.walk(root):
        # how many levels below root we are
        depth = folder[len(root):].count(os.sep)
        # if this folder has the file, remember it
        if file_name in files:
            # add it to the list
            found.append(folder)
        # do not go deeper than max_depth
        if depth >= max_depth:
            # clearing the list stops os.walk from going down
            subfolders[:] = []
    # give back the list
    return found


# look for run folders: they contain results.csv AND plan.csv
run_candidates = [f for f in folders_containing(INPUT_ROOT, "results.csv", 7) if os.path.isfile(os.path.join(f, "plan.csv"))]
# if none was found, maybe only the zip was uploaded
if len(run_candidates) == 0:
    # walk through the inputs looking for run_*.zip files
    for folder, subfolders, files in os.walk(INPUT_ROOT):
        # check each file name
        for file_name in files:
            # a zip made by notebook 02's Step 13
            if file_name.startswith("run_") and file_name.endswith(".zip"):
                # say what we found
                print("Unzipping", os.path.join(folder, file_name))
                # unpack it into the working folder
                zipfile.ZipFile(os.path.join(folder, file_name)).extractall(WORK_DIR + "/unzipped")
    # search the unpacked files for run folders
    run_candidates = [f for f in folders_containing(WORK_DIR + "/unzipped", "results.csv", 4) if os.path.isfile(os.path.join(f, "plan.csv"))]
# stop if there is still nothing
if len(run_candidates) == 0:
    # stop with a clear message
    raise RuntimeError("No run folder (results.csv + plan.csv) found under /kaggle/input. Did you Add Input > your notebook-02 output?")

# folder names look like run_YYYYMMDD_HHMM, so sorting by name puts the newest last
run_candidates = sorted(run_candidates, key=os.path.basename)
# use the newest run
SOURCE_RUN_DIR = run_candidates[-1]
# its name, e.g. run_20261024_1530
RUN_NAME = os.path.basename(SOURCE_RUN_DIR)
# where our writable copy goes
RUN_DIR = WORK_DIR + "/runs/" + RUN_NAME
# copy it only if we have not copied it already (keeps earlier answers if you re-run)
if not os.path.isdir(RUN_DIR):
    # copy the whole run folder
    shutil.copytree(SOURCE_RUN_DIR, RUN_DIR)
# the figures folder inside the copy
FIGURES_DIR = RUN_DIR + "/figures"
# make sure it exists
os.makedirs(FIGURES_DIR, exist_ok=True)
# report what we found
print("Run folder found:", SOURCE_RUN_DIR)
# report if there were several
if len(run_candidates) > 1:
    # list the others (we use the newest)
    print("  (other runs found, not used:", [os.path.basename(f) for f in run_candidates[:-1]], ")")
# report where we work
print("Working copy     :", RUN_DIR)
# print a title for the file counts
print("Files per folder :")
# go through the expected sub-folders
for sub in ["crops_src", "crops_edit", "tampered", "tampered_jpg", "originals", "masks/box", "masks/diff"]:
    # the full path
    sub_path = RUN_DIR + "/" + sub
    # print the count, or MISSING
    print(f"  {sub:13} {len(os.listdir(sub_path)) if os.path.isdir(sub_path) else 'MISSING'}")

# find the PATH_TO_DATA500_DATASET pages folder (the one holding the *_epdf bank folders)
DATA_DIR = None
# look in the dataset folder and one level below it
for candidate in [DATA_ROOT] + ([os.path.join(DATA_ROOT, n) for n in os.listdir(DATA_ROOT)] if os.path.isdir(DATA_ROOT) else []):
    # it must be a folder with at least one *_epdf sub-folder
    if os.path.isdir(candidate) and any(n.endswith("_epdf") for n in os.listdir(candidate)):
        # use it
        DATA_DIR = candidate
        # stop looking
        break
# report it
print("PATH_TO_DATA500_DATASET pages    :", DATA_DIR if DATA_DIR else "NOT FOUND - Add Input > PATH_TO_DATA500_DATASET (needed for the blind test)")

# collect the evaluation settings
eval_config = {
    "seed": SEED,
    "source_run": SOURCE_RUN_DIR,
    "n_blind_each": N_BLIND_EACH,
    "n_worst": N_WORST,
    "failure_types": FAILURE_TYPES,
    "ela_quality": ELA_QUALITY,
    "ela_ratio_threshold": ELA_RATIO_THRESHOLD,
}
# save them next to notebook 02's config.json
with open(RUN_DIR + "/eval_config.json", "w") as config_file:
    # write them with nice indentation
    json.dump(eval_config, config_file, indent=2)
# report the GPU
print("GPU              :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - slower but works)")

## Step 2 - Install the two tools we need
This notebook does not need TextCtrl at all; it only grades its output. We need **EasyOCR** (to read the edited
words back) and **scikit-image** (which has ready-made SSIM and PSNR functions). Same rule as before: install only if
missing.

**Expected output:** one line per package (`already installed` or `installed OK`).

In [ ]:
# import sys so we know which Python is running
import sys
# import subprocess so we can run pip
import subprocess
# import importlib.util so we can ask "is this package installed?"
import importlib.util

# the packages: (name for pip, name for import, what it is for)
packages = [
    ("easyocr", "easyocr", "reads the text in a picture and tells us where each word is"),
    ("scikit-image", "skimage", "image maths toolbox; we use its SSIM and PSNR functions"),
]
# go through each package
for pip_name, import_name, eli5 in packages:
    # explain it in one line
    print(f"- {pip_name}: {eli5}")
    # if Python can already find it, skip it
    if importlib.util.find_spec(import_name) is not None:
        # report it
        print("    already installed")
    # otherwise install it
    else:
        # run pip quietly
        run = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], capture_output=True, text=True)
        # report the result (and the end of the error if it failed)
        print("    installed OK" if run.returncode == 0 else "    FAILED:\n" + run.stderr[-1500:])

## Step 3 - Did the text come out right, and was the style kept? (P8)
A good forgery has to do two things at once: **say the new thing** (the target text) and **look like it was always
there** (same font, colour, background). We measure the first with OCR (a machine reading the edited crop) and the
second by comparing the edited crop with the original crop.

### Metric 1 - OCR accuracy
- **WHAT:** the share of forged crops where EasyOCR reads exactly the target text (ignoring capital letters).
- **WHY:** if a machine reads the new text, the forgery "says" what we wanted. A bank's automatic system would read it too.
- **HOW:** read the edited crop with EasyOCR (letters and digits only), lower-case both, compare.
- **Formula:** accuracy = (number of exact matches) / (number of crops).
- **Example:** target `4821`. OCR reads `4821` -> match (1). OCR reads `4B21` -> no match (0). 2 crops, 1 match -> 50%.

### Metric 2 - NED (normalised edit distance)
- **WHAT:** how many single-character fixes (insert, delete, swap) turn the OCR reading into the target, divided by the
  longer length. 0 = perfect, 1 = completely wrong.
- **WHY:** exact match is all-or-nothing; NED gives partial credit. `4B21` is much closer to `4821` than `zzzz` is.
- **HOW:** the classic edit-distance table (dynamic programming), then divide.
- **Formula:** NED = edit_distance(ocr, target) / max(len(ocr), len(target)).
- **Example:** `4B21` vs `4821`: 1 swap / 4 characters = **0.25**.

### Metric 3 - SSIM (structural similarity)
- **WHAT:** a score from -1 to 1 for how similar two pictures are in brightness, contrast and structure (1 = identical).
- **WHY:** our stand-in for "style kept": same background, colours and stroke thickness. The text DID change, so it
  can never be 1; a higher score means the edit looks more like the original crop.
- **HOW:** compare the grey-level source crop and edited crop in small 7x7 windows, then average.
- **Formula:** SSIM(x, y) = ((2 mu_x mu_y + C1)(2 sigma_xy + C2)) / ((mu_x^2 + mu_y^2 + C1)(sigma_x^2 + sigma_y^2 + C2)),
  where mu = mean, sigma^2 = variance, sigma_xy = covariance (just like a correlation coefficient), C1, C2 = small constants.
- **Example:** the same crop twice -> 1.0. Same background but different digits -> maybe 0.6.

### Metric 4 - PSNR (peak signal-to-noise ratio)
- **WHAT:** the average pixel difference on a log (decibel) scale. Higher = closer to the original.
- **WHY:** the standard, simple pixel-level check that sits next to SSIM in image papers.
- **HOW:** mean squared error (MSE) between the two grey crops, then a log.
- **Formula:** PSNR = 10 * log10(255^2 / MSE), with MSE = mean((x - y)^2).
- **Example:** MSE = 100 -> 10 * log10(65025 / 100) = **28.1 dB**.

### Metric 5 - MSE (mean squared error)
- **WHAT:** the average of the squared pixel differences between the source crop and the edited crop (grey levels).
- **WHY:** the plain pixel-level number the TextCtrl paper also reports; PSNR is just MSE on a log scale.
- **HOW:** subtract the two grey crops pixel by pixel, square, average.
- **Formula:** MSE = mean((x - y)^2). Lower = closer to the original.
- **Example:** every pixel differs by 10 grey levels -> MSE = 100 (and PSNR = 28.1 dB, the example above).

### Metric 6 - Character accuracy
- **WHAT:** the share of the target word's characters that OCR got right. Word accuracy (Metric 1) is all-or-nothing;
  this gives partial credit per character.
- **WHY:** the paper reports both word accuracy and character accuracy (ACC). Our definition may differ in detail from theirs.
- **HOW:** count the single-character fixes needed (edit distance), divide by the target's length, subtract from 1.
- **Formula:** char_acc = max(0, 1 - edit_distance(ocr, target) / len(target)).
- **Example:** target `4821`, OCR `4B21`: 1 fix / 4 characters -> 1 - 0.25 = **75%**, while word accuracy is 0%.

**Not in this notebook (limitations for the slides):** the paper's **FID** needs thousands of images to be meaningful,
and the paper scores text with TextCtrl's own recognizer, while we use EasyOCR.

We also OCR the ORIGINAL crops: if EasyOCR misreads untouched words too, some "wrong text" is OCR's fault, not TextCtrl's.

**Expected output:** progress every 20 crops, a headline table (OCR accuracy, mean NED, character accuracy, SSIM, PSNR, MSE), the same split by
kind (number vs word) and by word length, and `figures/ned_ssim_histograms.png`.
**Write down:** the four headline numbers.

In [ ]:
# import time so we can time the OCR
import time
# import numpy for maths
import numpy as np
# import pandas for the results table
import pandas as pd
# import torch so EasyOCR can use the GPU
import torch
# import matplotlib for the histograms
import matplotlib.pyplot as plt
# import PIL so we can open crops
from PIL import Image
# import the ready-made SSIM and PSNR functions
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
# import EasyOCR (installed in Step 2)
import easyocr

# read results.csv, keeping the texts as text (so 0123 stays 0123)
results = pd.read_csv(RUN_DIR + "/results.csv", dtype={"source_text": str, "target_text": str})
# report how many forgeries there are
print("Forgeries to grade:", len(results))
# load EasyOCR for English (on the GPU if there is one)
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
# EasyOCR may only answer with letters and digits (the same characters TextCtrl can draw)
ALLOWED = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"


# define a helper that gives the left edge of an OCR piece (so we can sort pieces left to right)
def left_edge(piece):
    # the piece's box is 4 corner points; take the smallest x
    return min(point[0] for point in piece[0])


# define a helper that reads one crop and returns the text with no spaces
def read_crop(path):
    # ask EasyOCR for every piece of text in the crop
    pieces = reader.readtext(np.array(Image.open(path).convert("RGB")), allowlist=ALLOWED)
    # put the pieces in left-to-right order
    pieces = sorted(pieces, key=left_edge)
    # glue the texts together and drop spaces
    return "".join(piece[1] for piece in pieces).replace(" ", "")


# define a helper that counts single-character fixes between two words (classic edit distance)
def edit_distance(a, b):
    # the first row of the table: turning "" into the first j letters of b takes j inserts
    previous = list(range(len(b) + 1))
    # fill the table one letter of a at a time
    for i in range(1, len(a) + 1):
        # turning the first i letters of a into "" takes i deletes
        current = [i]
        # go through the letters of b
        for j in range(1, len(b) + 1):
            # swapping costs 0 if the letters already match, else 1
            swap_cost = 0 if a[i - 1] == b[j - 1] else 1
            # the cheapest of: delete, insert, swap
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + swap_cost))
        # this row becomes the previous row
        previous = current
    # the bottom-right cell is the answer
    return previous[-1]


# define a helper for NED: edit distance divided by the longer length
def ned(a, b):
    # two empty strings are identical
    if max(len(a), len(b)) == 0:
        # distance 0
        return 0.0
    # divide by the longer length so the answer is between 0 and 1
    return edit_distance(a, b) / max(len(a), len(b))


# define a helper for character accuracy: the share of the target's characters that OCR got right
def char_accuracy(read_text, target_text):
    # an empty target cannot be scored
    if len(target_text) == 0:
        # count it as perfect
        return 1.0
    # share of characters that did not need fixing, never below 0
    return max(0.0, 1 - edit_distance(read_text, target_text) / len(target_text))


# define a helper that returns (SSIM, PSNR, MSE) between two crops
def ssim_psnr(path_a, path_b):
    # the source crop in grey levels
    grey_a = np.array(Image.open(path_a).convert("L"))
    # the edited crop in grey levels, forced to exactly the same size
    grey_b = np.array(Image.open(path_b).convert("L").resize((grey_a.shape[1], grey_a.shape[0])))
    # SSIM needs an odd window no bigger than the crop's smallest side
    smallest_side = min(grey_a.shape)
    # use 7, or the largest odd number that fits
    window = min(7, smallest_side if smallest_side % 2 == 1 else smallest_side - 1)
    # very tiny crops cannot be scored
    if window < 3:
        # mark as missing
        ssim = float("nan")
    # otherwise compute SSIM
    else:
        # 255 = the range of grey levels
        ssim = structural_similarity(grey_a, grey_b, win_size=window, data_range=255)
    # compute PSNR
    psnr = peak_signal_noise_ratio(grey_a, grey_b, data_range=255)
    # compute MSE: the average squared pixel difference (decimals, so the squares cannot overflow)
    mse = float(np.mean((grey_a.astype(np.float64) - grey_b.astype(np.float64)) ** 2))
    # give back all three
    return float(ssim), float(psnr), mse


# only compute if this was not done in an earlier run of this cell
if "mse" not in results.columns:
    # empty lists, one per new column
    ocr_edited, ocr_source, text_correct, ned_scores, source_ocr_correct, ssim_scores, psnr_scores = [], [], [], [], [], [], []
    # two more lists for the extra metrics
    char_acc_scores, mse_scores = [], []
    # start the stopwatch
    start = time.time()
    # go through every forgery
    for index, row in enumerate(results.to_dict("records"), start=1):
        # the edited crop
        edited_path = RUN_DIR + "/" + row["crop_edit"]
        # the original crop
        source_path = RUN_DIR + "/crops_src/" + row["crop_name"]
        # what OCR reads in the edited crop
        read_edit = read_crop(edited_path)
        # what OCR reads in the original crop
        read_source = read_crop(source_path)
        # store the readings
        ocr_edited.append(read_edit)
        # store the readings
        ocr_source.append(read_source)
        # exact match with the target (ignoring capitals)
        text_correct.append(read_edit.lower() == row["target_text"].lower())
        # NED against the target (ignoring capitals)
        ned_scores.append(round(ned(read_edit.lower(), row["target_text"].lower()), 3))
        # character accuracy against the target (ignoring capitals)
        char_acc_scores.append(round(char_accuracy(read_edit.lower(), row["target_text"].lower()), 3))
        # does OCR read the untouched crop correctly? (OCR's own error rate)
        source_ocr_correct.append(read_source.lower() == row["source_text"].lower())
        # style scores
        ssim, psnr, mse = ssim_psnr(source_path, edited_path)
        # store SSIM
        ssim_scores.append(round(ssim, 4))
        # store PSNR
        psnr_scores.append(round(psnr, 2))
        # store MSE
        mse_scores.append(round(mse, 2))
        # progress every 20 crops
        if index % 20 == 0:
            # print it
            print(f"  graded {index}/{len(results)} ({time.time() - start:.0f} s)")
    # add the new columns
    results["ocr_edited"] = ocr_edited
    # add the new columns
    results["ocr_source"] = ocr_source
    # add the new columns
    results["text_correct"] = text_correct
    # add the new columns
    results["ned"] = ned_scores
    # add the new columns
    results["source_ocr_correct"] = source_ocr_correct
    # add the new columns
    results["ssim"] = ssim_scores
    # add the new columns
    results["psnr"] = psnr_scores
    # add the new columns
    results["char_acc"] = char_acc_scores
    # add the new columns
    results["mse"] = mse_scores
    # word length (for the split table)
    results["word_length"] = results["source_text"].str.len()
    # save the updated results.csv in the working copy
    results.to_csv(RUN_DIR + "/results.csv", index=False)
# otherwise the columns already exist
else:
    # say so
    print("Metrics already in results.csv - not recomputed")


# define a helper that makes the summary numbers for a group of rows
def summarise(group):
    # one row of the summary table
    return pd.Series({
        "n": len(group),
        "ocr_accuracy": group["text_correct"].mean(),
        "mean_ned": group["ned"].mean(),
        "mean_ssim": group["ssim"].mean(),
        "mean_psnr": group["psnr"].mean(),
        "mean_mse": group["mse"].mean(),
        "mean_char_acc": group["char_acc"].mean(),
    })


# define a helper that summarises each group of a column (e.g. each kind) and stacks the rows into one table
def summary_by(column):
    # one summary row per group
    rows = {}
    # go through each group
    for group_name, group in results.groupby(column):
        # summarise this group
        rows[group_name] = summarise(group)
    # stack the rows into a table
    return pd.DataFrame(rows).T


# print a title
print()
# print the headline title
print("=== Headline numbers (all forgeries) ===")
# print the overall summary, rounded
print(summarise(results).round(3).to_string())
# print OCR's accuracy on the UNTOUCHED crops (baseline)
print(f"OCR accuracy on the untouched source crops (baseline): {results['source_ocr_correct'].mean():.3f}")
# print a title
print()
# print the split-by-kind title
print("=== Split by kind ===")
# group by number / word and summarise
print(summary_by("kind").round(3).to_string())
# print a title
print()
# print the split-by-length title
print("=== Split by word length ===")
# group by length and summarise
print(summary_by("word_length").round(3).to_string())

# two histograms side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
# NED histogram, 10 bins from 0 to 1
axes[0].hist(results["ned"], bins=np.linspace(0, 1, 11), color="tab:blue", edgecolor="white")
# title
axes[0].set_title("NED between OCR reading and target text")
# x label
axes[0].set_xlabel("NED (0 = perfect, 1 = completely wrong)")
# y label
axes[0].set_ylabel("number of forged crops")
# SSIM histogram
axes[1].hist(results["ssim"].dropna(), bins=10, color="tab:orange", edgecolor="white")
# title
axes[1].set_title("SSIM between source crop and edited crop")
# x label
axes[1].set_xlabel("SSIM (1 = identical look)")
# y label
axes[1].set_ylabel("number of forged crops")
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/ned_ssim_histograms.png", dpi=150)
# show it
plt.show()

## Step 4 - The 2x2 grid: right text x kept style (P9, part 1)
One number cannot describe a forgery, so we sort every forgery into one of four boxes using the two questions from
Step 3: **is the text correct** (OCR reads the target exactly) and **is the style kept** (SSIM above the median of all
forgeries). Like a 2x2 table in economics (e.g. high/low inflation x high/low growth), it shows which combinations are
common. The dangerous box is "text correct AND style kept": forgeries that both say the new thing and blend in.
Note: "above the median" means about half the crops count as style-kept by construction; the interesting part is how
that half splits between correct and wrong text.

**Expected output:** a 2x2 count table and a figure with 2 example forgeries per box
(`figures/grid_2x2_examples.png`). **Write down:** the four counts.

In [ ]:
# import pandas for the count table
import pandas as pd
# import matplotlib for the figure
import matplotlib.pyplot as plt
# import PIL so we can open crops
from PIL import Image

# the median SSIM of all forgeries
median_ssim = results["ssim"].median()
# style is "kept" if the SSIM is above the median
results["style_kept"] = results["ssim"] > median_ssim
# save the new column
results.to_csv(RUN_DIR + "/results.csv", index=False)
# count the forgeries in each of the four boxes
grid = pd.crosstab(results["text_correct"].map({True: "text correct", False: "text wrong"}),
                   results["style_kept"].map({True: "style kept", False: "style not kept"}))
# print the median used
print(f"Median SSIM (the style cut-off): {median_ssim:.3f}")
# print the table
print(grid.to_string())

# the four boxes, in reading order: (text_correct, style_kept, label)
boxes = [(True, True, "text correct + style kept"), (True, False, "text correct + style NOT kept"),
         (False, True, "text wrong + style kept"), (False, False, "text wrong + style NOT kept")]
# one row per box, four columns: example 1 before/after, example 2 before/after
fig, axes = plt.subplots(4, 4, figsize=(11, 6.5), squeeze=False)
# fill one row per box
for r, (correct, kept, label) in enumerate(boxes):
    # the forgeries in this box
    in_box = results[(results["text_correct"] == correct) & (results["style_kept"] == kept)]
    # pick up to 2 of them with the seed
    examples = in_box.sample(n=min(2, len(in_box)), random_state=SEED).to_dict("records")
    # hide all four axes in this row first (empty boxes stay blank)
    for c in range(4):
        # hide this axis
        axes[r][c].axis("off")
    # write the box name on the left of the row
    axes[r][0].set_title(f"{label} (n={len(in_box)})", fontsize=9, loc="left")
    # draw each example as a before/after pair
    for e, example in enumerate(examples):
        # the source crop goes in column 2e
        axes[r][2 * e].imshow(Image.open(RUN_DIR + "/crops_src/" + example["crop_name"]).convert("RGB"))
        # the edited crop goes in column 2e + 1
        axes[r][2 * e + 1].imshow(Image.open(RUN_DIR + "/" + example["crop_edit"]).convert("RGB"))
        # label the edited crop with the target and what OCR read
        axes[r][2 * e + 1].set_xlabel(f"{example['source_text']}->{example['target_text']} | OCR: {example['ocr_edited']}", fontsize=8)
        # show that label even though the axis is hidden
        axes[r][2 * e + 1].axis("on")
        # remove the tick marks
        axes[r][2 * e + 1].set_xticks([])
        # remove the tick marks
        axes[r][2 * e + 1].set_yticks([])
# give the figure a title
fig.suptitle("2x2 grid: 2 examples per box (each pair = source crop | edited crop)", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/grid_2x2_examples.png", dpi=150)
# show it
plt.show()

## Step 5 - Blind test: can YOU tell real from fake? (P9, part 2)
Numbers are one judge; a human eye is another. We mix **10 random forged crops** with **10 untouched word crops from
pages we did NOT forge** (found with the same OCR rules, so they look like the same kind of word), shuffle them with
the seed, and show them one at a time. You type `r` (real) or `f` (fake). Chance level is 10/20, so a score near 10
means the forgeries fool you; near 20 means they are easy to spot.

**How to answer:** a text box appears under each crop; type `r` or `f` and press Enter.
Your answers are saved to `blind_test.csv` straight away at the end, and if that file exists the test is NOT repeated
(so re-running the notebook is safe).

**Expected output:** 20 small pictures with an answer box under each, then `Your score: X / 20`, how many fakes you
caught and how many reals you trusted. **Write down:** your score.

In [ ]:
# import os so we can check files
import os
# import re so we can apply the word rule
import re
# import random so the shuffle follows the seed
import random
# import numpy so we can hand pages to EasyOCR
import numpy as np
# import pandas so we can save the answers
import pandas as pd
# import matplotlib so we can show each crop
import matplotlib.pyplot as plt
# import Path so page paths with spaces are handled safely
from pathlib import Path
# import PIL so we can open and crop pages
from PIL import Image

# where the answers are saved
blind_path = RUN_DIR + "/blind_test.csv"
# if the test was already done, just report it
if os.path.exists(blind_path):
    # read the saved answers
    blind = pd.read_csv(blind_path)
    # say so
    print("blind_test.csv already exists - not asking again")
# if we are not in interactive mode, skip
elif not INTERACTIVE:
    # nothing to report
    blind = None
    # say so
    print("INTERACTIVE is False - blind test skipped")
# if the PATH_TO_DATA500_DATASET pages are missing, we cannot make real crops
elif DATA_DIR is None:
    # nothing to report
    blind = None
    # say so
    print("PATH_TO_DATA500_DATASET not found - Add Input > PATH_TO_DATA500_DATASET, re-run Step 1, then this cell")
# otherwise run the test
else:
    # a random generator that starts from our seed
    rng = random.Random(SEED)
    # the folder for the real crops
    real_dir = RUN_DIR + "/blind_real_crops"
    # make it
    os.makedirs(real_dir, exist_ok=True)
    # every page in PATH_TO_DATA500_DATASET, in a fixed order
    all_pages = sorted([p.relative_to(DATA_DIR) for p in Path(DATA_DIR).rglob("*.png")])
    # the pages we forged (we must not use them)
    used_pages = set(results["page_path"])
    # the pages we did NOT forge
    unused_pages = [p for p in all_pages if str(p) not in used_pages]
    # shuffle them with the seed
    rng.shuffle(unused_pages)
    # the word rule
    word_pattern = re.compile(WORD_REGEX)
    # the list of real crop files
    real_crops = []
    # go through unused pages until we have enough real crops
    for rel_path in unused_pages:
        # stop when we have enough
        if len(real_crops) >= N_BLIND_EACH:
            # leave the loop
            break
        # open the page
        page_img = Image.open(Path(DATA_DIR) / rel_path).convert("RGB")
        # find words with EasyOCR
        detections = reader.readtext(np.array(page_img))
        # keep the words that pass the same rules as notebook 02
        usable = []
        # check each word
        for bbox, text, confidence in detections:
            # the box's left and right
            x1, x2 = int(min(p[0] for p in bbox)), int(max(p[0] for p in bbox))
            # the box's top and bottom
            y1, y2 = int(min(p[1] for p in bbox)), int(max(p[1] for p in bbox))
            # all three rules
            if word_pattern.match(text) and confidence >= MIN_CONFIDENCE and (y2 - y1) >= MIN_BOX_HEIGHT:
                # keep it
                usable.append((x1, y1, x2, y2))
        # skip pages with no usable word
        if len(usable) == 0:
            # next page
            continue
        # pick one word with the seed
        x1, y1, x2, y2 = rng.choice(usable)
        # the crop file name
        real_name = f"real_{len(real_crops) + 1:02d}.png"
        # the page size, so the padding never goes past the edge (a black border would give the answer away)
        page_width, page_height = page_img.size
        # crop with the same padding as notebook 02 and save it
        page_img.crop((max(0, x1 - CROP_PADDING), max(0, y1 - CROP_PADDING), min(page_width, x2 + CROP_PADDING), min(page_height, y2 + CROP_PADDING))).save(real_dir + "/" + real_name)
        # remember it
        real_crops.append("blind_real_crops/" + real_name)
    # pick the fake crops with the seed
    fake_crops = list(results.sample(n=N_BLIND_EACH, random_state=SEED)["crop_edit"])
    # every item: (file, truth)
    items = [(f, "real") for f in real_crops] + [(f, "fake") for f in fake_crops]
    # shuffle the items with the seed
    rng.shuffle(items)
    # the answers
    answers = []
    # show the items one at a time
    for number, (rel_file, truth) in enumerate(items, start=1):
        # a small figure for this crop
        fig, ax = plt.subplots(figsize=(4, 1.4))
        # draw the crop
        ax.imshow(Image.open(RUN_DIR + "/" + rel_file).convert("RGB"))
        # label it with its number only (no hint)
        ax.set_title(f"Item {number} of {len(items)}", fontsize=10)
        # hide the axis numbers
        ax.axis("off")
        # show it now
        plt.show()
        # ask until we get r or f
        answer = ""
        # keep asking while the answer is not r or f
        while answer not in ["r", "f"]:
            # read what you typed
            answer = input("real or fake? type r or f, then Enter: ").strip().lower()
        # remember the answer
        answers.append({"item": number, "file": rel_file, "truth": truth, "answer": "real" if answer == "r" else "fake"})
    # turn the answers into a table
    blind = pd.DataFrame(answers)
    # mark each answer right or wrong
    blind["correct"] = blind["truth"] == blind["answer"]
    # save the answers
    blind.to_csv(blind_path, index=False)

# if there are answers, report the score
if blind is not None:
    # the total score
    print(f"Your score: {int(blind['correct'].sum())} / {len(blind)}  (chance = {len(blind) // 2})")
    # how many fakes were caught
    print(f"Fakes you caught   : {int(((blind['truth'] == 'fake') & blind['correct']).sum())} / {int((blind['truth'] == 'fake').sum())}")
    # how many reals were trusted
    print(f"Reals you trusted  : {int(((blind['truth'] == 'real') & blind['correct']).sum())} / {int((blind['truth'] == 'real').sum())}")

## Step 6 - Failure gallery: tag the 8 worst forgeries (P9, part 3)
Averages hide HOW things fail. We take the 8 forgeries with the highest NED (where OCR's reading is furthest from the
target) and you give each ONE failure type. Counting the tags tells us the most common failure, which is more useful
for the slides than "NED was 0.4".

Failure types: **1** wrong text, **2** blurry, **3** colour/background mismatch, **4** squashed long word,
**5** box misaligned, **6** other. Type the number and press Enter. Saved to `failure_tags.csv`; not asked again on re-runs.

**Expected output:** 8 before/after pairs, each with an answer box, then a count of each failure type and the most
common one; also `figures/failure_gallery.png`. **Write down:** the most common failure type.

In [ ]:
# import os so we can check files
import os
# import pandas so we can save the tags
import pandas as pd
# import matplotlib so we can show the pairs
import matplotlib.pyplot as plt
# import PIL so we can open crops
from PIL import Image

# where the tags are saved
tags_path = RUN_DIR + "/failure_tags.csv"
# the 8 worst forgeries by NED
worst = results.sort_values("ned", ascending=False).head(N_WORST).to_dict("records")

# one row per forgery, two columns: before | after (saved as the gallery figure)
fig, axes = plt.subplots(len(worst), 2, figsize=(6, 1.2 * len(worst)), squeeze=False)
# fill one row per forgery
for i, row in enumerate(worst):
    # the original crop
    axes[i][0].imshow(Image.open(RUN_DIR + "/crops_src/" + row["crop_name"]).convert("RGB"))
    # label it
    axes[i][0].set_title(f"#{i + 1} source: {row['source_text']}", fontsize=8)
    # hide the axis numbers
    axes[i][0].axis("off")
    # the edited crop
    axes[i][1].imshow(Image.open(RUN_DIR + "/" + row["crop_edit"]).convert("RGB"))
    # label it with the target, the OCR reading and the NED
    axes[i][1].set_title(f"target: {row['target_text']} | OCR: {row['ocr_edited']} | NED {row['ned']:.2f}", fontsize=8)
    # hide the axis numbers
    axes[i][1].axis("off")
# give the figure a title
fig.suptitle(f"The {len(worst)} worst forgeries by NED", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/failure_gallery.png", dpi=150)
# show it
plt.show()

# if the tags already exist, just load them
if os.path.exists(tags_path):
    # read them
    tags = pd.read_csv(tags_path)
    # say so
    print("failure_tags.csv already exists - not asking again")
# if we are not in interactive mode, skip
elif not INTERACTIVE:
    # nothing to report
    tags = None
    # say so
    print("INTERACTIVE is False - tagging skipped")
# otherwise ask for one tag per forgery
else:
    # print the menu of failure types
    print("Failure types:", ", ".join(f"{n}={name}" for n, name in enumerate(FAILURE_TYPES, start=1)))
    # the list of tags
    tag_rows = []
    # one question per forgery (the numbers match the gallery above)
    for i, row in enumerate(worst, start=1):
        # start with an invalid answer
        choice = ""
        # keep asking until the answer is a number from 1 to 6
        while choice not in [str(n) for n in range(1, len(FAILURE_TYPES) + 1)]:
            # read what you typed
            choice = input(f"#{i} {row['source_text']} -> {row['target_text']} (OCR: {row['ocr_edited']}). Failure type 1-{len(FAILURE_TYPES)}: ").strip()
        # remember the tag
        tag_rows.append({"crop_name": row["crop_name"], "source_text": row["source_text"], "target_text": row["target_text"],
                         "ocr_edited": row["ocr_edited"], "ned": row["ned"], "failure_type": FAILURE_TYPES[int(choice) - 1]})
    # turn the tags into a table
    tags = pd.DataFrame(tag_rows)
    # save them
    tags.to_csv(tags_path, index=False)

# if there are tags, count them
if tags is not None:
    # count each failure type
    counts = tags["failure_type"].value_counts()
    # print the counts
    print(counts.to_string())
    # print the most common one
    print("Most common failure type:", counts.index[0])

## Step 7 - Can the Week 1 tools catch it? ELA and SRM (P10)
Steps 3-6 asked "is it convincing?". Now we ask "does it leave a trace a simple forensic tool can see?"

### ELA (Error Level Analysis)
- **WHAT:** re-save the page as JPEG (quality 90) and look at how much each pixel changes. Parts of an image with a
  different history (pasted in, re-drawn) often change by a different amount than the rest.
- **WHY:** the classic first check from Week 1; if it lights up our edit, the forgery is easy to catch.
- **HOW:** ELA(pixel) = |page - JPEG90(page)| (the strongest of the three colour channels). For display we brighten it x20.
- **ELA ratio:** mean ELA inside the box mask / mean ELA outside it. ~1 = ELA cannot see the edit; we call > 1.5 "lights up".
- **Example:** inside 6.0, outside 3.0 -> ratio **2.0** (stands out). Inside 3.1, outside 3.0 -> **1.03** (invisible).

### SRM (noise residual)
- **WHAT:** a high-pass filter that removes the picture's content and keeps only the fine "noise" texture.
- **WHY:** a camera or scanner leaves a consistent noise pattern; a region made by a different process (here, a
  diffusion model) often has a different texture, even if it looks perfect to the eye.
- **HOW / formula:** residual = |K * grey page|, where K is the classic 5x5 SRM "KV" kernel and * means sliding the
  kernel over the page (a convolution).

**Our expectation (a reflection point):** these pages were never JPEG-compressed (all PNG), so ELA has little
compression history to compare; the paste BOUNDARY (a sharp rectangle where texture changes) may show up even when
the text itself looks perfect. As a control, we also compute the same ratio on the ORIGINAL page with the same box:
that shows what "no edit" looks like.

**Expected output:** a figure of 4 pages zoomed in around the edit: original | tampered | ELA | SRM | box mask
(`figures/ela_srm_examples.png`), a histogram of ELA ratios for tampered vs original pages (`figures/ela_ratio_histogram.png`),
and the share of pages with ratio > 1.5. **Write down:** does the edit light up, and the percentage.

In [ ]:
# import io so we can JPEG-compress in memory (no extra files)
import io
# import time so we can time the loop
import time
# import numpy for the maths
import numpy as np
# import cv2 for the SRM filter
import cv2
# import matplotlib for the figures
import matplotlib.pyplot as plt
# import PIL so we can open and re-save pages
from PIL import Image

# the classic 5x5 SRM "KV" kernel (divided by 12)
SRM_KERNEL = np.array([[-1, 2, -2, 2, -1],
                       [2, -6, 8, -6, 2],
                       [-2, 8, -12, 8, -2],
                       [2, -6, 8, -6, 2],
                       [-1, 2, -2, 2, -1]], dtype=np.float32) / 12.0


# define a helper that makes the ELA map of a page
def ela_map(page_rgb):
    # an in-memory file
    buffer = io.BytesIO()
    # save the page as JPEG quality 90 into it
    page_rgb.save(buffer, "JPEG", quality=ELA_QUALITY)
    # rewind to the start of the in-memory file
    buffer.seek(0)
    # open the re-saved version
    resaved = Image.open(buffer).convert("RGB")
    # absolute difference per colour channel (signed numbers so we can subtract)
    difference = np.abs(np.array(page_rgb, dtype=np.int16) - np.array(resaved, dtype=np.int16))
    # keep the strongest channel at each pixel
    return difference.max(axis=2).astype(np.float32)


# define a helper that makes the SRM noise residual of a page
def srm_map(page_rgb):
    # the page in grey levels as decimals
    grey = np.array(page_rgb.convert("L"), dtype=np.float32)
    # slide the kernel over the page and keep the size of the answer
    return np.abs(cv2.filter2D(grey, -1, SRM_KERNEL))


# define a helper for the ELA ratio: mean inside the mask / mean outside
def ela_ratio(ela, mask):
    # mean ELA where the mask is white (the edit)
    inside = ela[mask > 0].mean()
    # mean ELA everywhere else
    outside = ela[mask == 0].mean()
    # divide (with a tiny floor so we never divide by 0)
    return float(inside / max(outside, 1e-6))


# only compute if this was not done in an earlier run of this cell
if "ela_ratio_tampered" not in results.columns:
    # ratios for the forged pages
    ratios_tampered = []
    # ratios for the untouched pages (control)
    ratios_original = []
    # start the stopwatch
    start = time.time()
    # go through every forged page
    for index, row in enumerate(results.to_dict("records"), start=1):
        # the box mask (white = edit)
        mask = np.array(Image.open(RUN_DIR + "/" + row["mask_box"]))
        # the forged page
        tampered = Image.open(RUN_DIR + "/" + row["tampered_png"]).convert("RGB")
        # the untouched page
        original = Image.open(RUN_DIR + "/" + row["original_png"]).convert("RGB")
        # ELA ratio of the forged page
        ratios_tampered.append(round(ela_ratio(ela_map(tampered), mask), 3))
        # ELA ratio of the untouched page, same box (control)
        ratios_original.append(round(ela_ratio(ela_map(original), mask), 3))
        # progress every 25 pages
        if index % 25 == 0:
            # print it
            print(f"  ELA on {index}/{len(results)} pages ({time.time() - start:.0f} s)")
    # add the columns
    results["ela_ratio_tampered"] = ratios_tampered
    # add the columns
    results["ela_ratio_original"] = ratios_original
    # save the updated results.csv
    results.to_csv(RUN_DIR + "/results.csv", index=False)

# share of forged pages where the edit lights up
share_tampered = (results["ela_ratio_tampered"] > ELA_RATIO_THRESHOLD).mean()
# the same share for untouched pages (false alarms)
share_original = (results["ela_ratio_original"] > ELA_RATIO_THRESHOLD).mean()
# print the headline
print(f"Pages where ELA ratio > {ELA_RATIO_THRESHOLD}: tampered {share_tampered:.0%}  |  untouched (control) {share_original:.0%}")
# print the medians
print(f"Median ELA ratio: tampered {results['ela_ratio_tampered'].median():.2f}  |  untouched {results['ela_ratio_original'].median():.2f}")

# histogram of ELA ratios, tampered vs untouched
fig, ax = plt.subplots(figsize=(7, 3.8))
# shared bins for both
bins = np.linspace(0, max(3.0, float(results["ela_ratio_tampered"].max())), 31)
# the tampered pages
ax.hist(results["ela_ratio_tampered"], bins=bins, alpha=0.7, label="tampered page", color="tab:red")
# the untouched pages
ax.hist(results["ela_ratio_original"], bins=bins, alpha=0.5, label="untouched page (same box)", color="tab:gray")
# the "lights up" line
ax.axvline(ELA_RATIO_THRESHOLD, color="black", linestyle="--", label=f"threshold {ELA_RATIO_THRESHOLD}")
# title
ax.set_title("ELA ratio: mean ELA inside the edited box / mean outside")
# x label
ax.set_xlabel("ELA ratio (about 1 = edit invisible to ELA)")
# y label
ax.set_ylabel("number of pages")
# legend
ax.legend(fontsize=8)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/ela_ratio_histogram.png", dpi=150)
# show it
plt.show()

# the example pages: the first N_ELA_EXAMPLES rows
examples = results.head(N_ELA_EXAMPLES).to_dict("records")
# one row per page, five columns
fig, axes = plt.subplots(len(examples), 5, figsize=(14, 2.0 * len(examples)), squeeze=False)
# the column titles
titles = ["original", "tampered", "ELA (x20)", "SRM residual", "box mask"]
# fill one row per page
for i, row in enumerate(examples):
    # the zoom window around the box: left, top, right, bottom
    window = (max(0, int(row["x1"]) - ZOOM_MARGIN), max(0, int(row["y1"]) - ZOOM_MARGIN), int(row["x2"]) + ZOOM_MARGIN, int(row["y2"]) + ZOOM_MARGIN)
    # the untouched page
    original = Image.open(RUN_DIR + "/" + row["original_png"]).convert("RGB")
    # the forged page
    tampered = Image.open(RUN_DIR + "/" + row["tampered_png"]).convert("RGB")
    # ELA of the forged page, brightened for our eyes and capped at 255
    ela_display = np.clip(ela_map(tampered) * ELA_AMPLIFY, 0, 255).astype(np.uint8)
    # SRM of the forged page, stretched to 0-255 for display
    srm = srm_map(tampered)
    # stretch so the brightest 1% becomes white
    srm_display = np.clip(srm / max(np.percentile(srm, 99), 1e-6) * 255, 0, 255).astype(np.uint8)
    # the box mask
    mask = Image.open(RUN_DIR + "/" + row["mask_box"])
    # the five pictures for this row, all cut to the zoom window
    pictures = [original.crop(window), tampered.crop(window), Image.fromarray(ela_display).crop(window),
                Image.fromarray(srm_display).crop(window), mask.crop(window)]
    # draw each picture
    for col, picture in enumerate(pictures):
        # grey colour map for the single-channel pictures
        axes[i][col].imshow(picture, cmap="gray")
        # hide the axis numbers
        axes[i][col].axis("off")
        # put the column title on the top row only
        if i == 0:
            # write the title
            axes[i][col].set_title(titles[col], fontsize=10)
    # label the row with the edit and its ELA ratio
    axes[i][0].text(0, -8, f"{row['source_text']}->{row['target_text']}  ELA ratio {row['ela_ratio_tampered']:.2f}", fontsize=8)
# give the figure a title
fig.suptitle("Week 1 tools on 4 forged pages (zoomed in around the edit)", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/ela_srm_examples.png", dpi=150)
# show it
plt.show()

## Step 7b - Detection metrics: precision, recall, F1 (feedback)
Step 7 asked "does ELA light up the edit?". Here we score ELA the way a fraud-detection system is scored: it looks at a
page and says **forged** or **not forged**, and we count how often it is right. We use the 100 forged pages and the 100
untouched pages (the control from Step 7). A page is *flagged* when its ELA ratio is above 1.5 (our own threshold).

Four outcomes: **TP** forged page flagged, **FN** forged page missed, **FP** untouched page wrongly flagged (a false
alarm), **TN** untouched page correctly left alone.

### Precision
- **WHAT:** of the pages the detector flagged, the share that really are forged.
- **WHY:** false alarms cost money (a bank reviewing a clean statement). Low precision = many false alarms.
- **Formula:** precision = TP / (TP + FP). **Example:** flags 8 pages, 6 really forged -> 6/8 = **75%**.

### Recall
- **WHAT:** of the pages that really are forged, the share the detector caught.
- **WHY:** a missed forgery is the expensive mistake. Low recall = forgeries slip through.
- **Formula:** recall = TP / (TP + FN). **Example:** 10 forged pages, 6 caught -> 6/10 = **60%**.

### F1 score
- **WHAT:** one number that combines precision and recall (their harmonic mean, which is dragged down by whichever is smaller).
- **WHY:** a detector that flags everything has perfect recall but terrible precision; F1 punishes that.
- **Formula:** F1 = 2 x precision x recall / (precision + recall). **Example:** 0.75 and 0.60 -> 2 x 0.45 / 1.35 = **0.667**.

Same metrics will be used in Week 5 to score ADCD-Net, pixel by pixel against the box masks.
Note this scores the detector (ELA), not TextCtrl: a LOW score here means the forgery is hard for ELA to see.

**Expected output:** a 2x2 table of TP/FN/FP/TN, then precision, recall, F1 and accuracy, and a bar chart saved as
`figures/ela_detection_metrics.png`. **Write down:** precision, recall and F1.

In [ ]:
# import pandas for the confusion table
import pandas as pd
# import matplotlib for the bar chart
import matplotlib.pyplot as plt

# stop early if Step 7 did not run (it creates the ELA ratio columns)
if "ela_ratio_tampered" not in results.columns:
    # stop with a clear message
    raise RuntimeError("Run Step 7 first: it creates the ela_ratio columns")

# a page is "flagged" when its ELA ratio is above this (the same threshold as Step 7)
threshold = ELA_RATIO_THRESHOLD
# forged pages that ELA flagged (true positives)
true_positives = int((results["ela_ratio_tampered"] > threshold).sum())
# forged pages that ELA missed (false negatives)
false_negatives = int((results["ela_ratio_tampered"] <= threshold).sum())
# untouched pages that ELA wrongly flagged (false positives = false alarms)
false_positives = int((results["ela_ratio_original"] > threshold).sum())
# untouched pages that ELA correctly left alone (true negatives)
true_negatives = int((results["ela_ratio_original"] <= threshold).sum())

# precision: of the flagged pages, the share that are really forged
precision = 0.0
# only divide if something was flagged
if true_positives + false_positives > 0:
    # TP / (TP + FP)
    precision = true_positives / (true_positives + false_positives)
# recall: of the forged pages, the share that were caught
recall = 0.0
# only divide if there are forged pages
if true_positives + false_negatives > 0:
    # TP / (TP + FN)
    recall = true_positives / (true_positives + false_negatives)
# F1: the harmonic mean of precision and recall
f1 = 0.0
# only divide if at least one of them is above 0
if precision + recall > 0:
    # 2 x P x R / (P + R)
    f1 = 2 * precision * recall / (precision + recall)
# accuracy: the share of all pages classified correctly
accuracy = (true_positives + true_negatives) / (true_positives + false_negatives + false_positives + true_negatives)

# the 2x2 table of outcomes
confusion = pd.DataFrame([[true_positives, false_negatives], [false_positives, true_negatives]],
                         index=["actually forged (100)", "actually untouched (100)"],
                         columns=["flagged by ELA", "not flagged"])
# print a title
print(f"=== ELA as a detector (flag a page when ELA ratio > {threshold}) ===")
# print the table
print(confusion.to_string())
# print the metrics
print(f"Precision: {precision:.3f}   Recall: {recall:.3f}   F1: {f1:.3f}   Accuracy: {accuracy:.3f}")
# keep the numbers so Step 8 can put them in headline.json
ela_detection = {"threshold": threshold, "tp": true_positives, "fn": false_negatives, "fp": false_positives, "tn": true_negatives,
                 "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3), "accuracy": round(accuracy, 3)}

# a bar chart of the four numbers
fig, ax = plt.subplots(figsize=(5.5, 3.6))
# the names and values of the bars
names = ["precision", "recall", "F1", "accuracy"]
# the heights of the bars
values = [precision, recall, f1, accuracy]
# draw the bars
bars = ax.bar(names, values, color=["tab:blue", "tab:orange", "tab:green", "tab:gray"])
# write each value on top of its bar
for bar, value in zip(bars, values):
    # put the number just above the bar
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.2f}", ha="center", fontsize=10)
# the scores run from 0 to 1
ax.set_ylim(0, 1.1)
# y label
ax.set_ylabel("score (0 to 1, higher = better detection)")
# title
ax.set_title(f"ELA as a page-level detector (ratio > {threshold}), 100 forged + 100 untouched pages", fontsize=9)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/ela_detection_metrics.png", dpi=150)
# show it
plt.show()

## Step 8 - Headline numbers and a zip to download
This collects every number the slides need (P11) in one place, saves them as `headline.json`, and zips the whole
evaluated run folder (notebook 02's dataset + the new columns, answers and figures). This zip is also the dataset for
Week 5 (ADCD-Net). Download it from the file browser: folder icon > `working` > `<run name>_evaluated.zip`.

**Expected output:** the headline numbers, the list of figures, and `Zip saved: ... (size)`.

In [ ]:
# import os so we can list figures and read the zip size
import os
# import json so we can save the headline numbers
import json
# import shutil so we can make the zip
import shutil

# collect the headline numbers
headline = {
    "forgeries": int(len(results)),
    "ocr_accuracy": round(float(results["text_correct"].mean()), 3),
    "mean_ned": round(float(results["ned"].mean()), 3),
    "mean_ssim": round(float(results["ssim"].mean()), 3),
    "mean_psnr_db": round(float(results["psnr"].mean()), 2),
    "mean_mse": round(float(results["mse"].mean()), 1),
    "mean_char_accuracy": round(float(results["char_acc"].mean()), 3),
    "ocr_accuracy_on_untouched_crops": round(float(results["source_ocr_correct"].mean()), 3),
    "grid_2x2": {f"{'text correct' if c else 'text wrong'} / {'style kept' if k else 'style not kept'}":
                 int(((results["text_correct"] == c) & (results["style_kept"] == k)).sum())
                 for c in [True, False] for k in [True, False]},
    "ela_share_above_threshold_tampered": round(float((results["ela_ratio_tampered"] > ELA_RATIO_THRESHOLD).mean()), 3),
    "ela_share_above_threshold_untouched": round(float((results["ela_ratio_original"] > ELA_RATIO_THRESHOLD).mean()), 3),
}
# add the blind-test score if the test was done
if globals().get("blind") is not None:
    # score out of 20
    headline["blind_test_score"] = f"{int(blind['correct'].sum())} / {len(blind)}"
# add the most common failure type if the tags exist
if globals().get("tags") is not None:
    # the most common tag
    headline["most_common_failure"] = str(tags["failure_type"].value_counts().index[0])
# add the ELA detection metrics (precision, recall, F1) if Step 7b ran
if globals().get("ela_detection") is not None:
    # store them
    headline["ela_detection_page_level"] = ela_detection
# save the headline numbers
with open(RUN_DIR + "/headline.json", "w") as headline_file:
    # write them with nice indentation
    json.dump(headline, headline_file, indent=2)
# print them
print(json.dumps(headline, indent=2))
# list the figures made so far
print("Figures:", sorted(os.listdir(FIGURES_DIR)))

# zip the evaluated run folder into /kaggle/working
zip_path = shutil.make_archive(WORK_DIR + "/" + RUN_NAME + "_evaluated", "zip", root_dir=WORK_DIR + "/runs", base_dir=RUN_NAME)
# print where it is and how big it is
print(f"Zip saved: {zip_path} ({os.path.getsize(zip_path) / (1024 * 1024):.0f} MB)")

In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/run_20261024_0712_evaluated.zip")

if zip_path.is_file():
    os.chdir("/kaggle/working")
    display(FileLink(
        zip_path.name,
        result_html_prefix="Download your evaluated ZIP: "
    ))
else:
    print(f"File not found: {zip_path}")